In [1]:
import numpy as np
import tensorflow as tf
import tensorflow.keras as keras
import pickle

In [2]:
from tensorflow.compat.v1 import ConfigProto
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
config=tf.compat.v1.ConfigProto()
config.gpu_options.per_process_gpu_memory_fraction = 0.8
config.gpu_options.allow_growth=True
sess=tf.compat.v1.Session(config=config) 

In [3]:
from load_network import load_VGG, load_Res, load_ViT, load_Mix
from noise import PGDRegressionAdversaryL2
from train import WarmUpCosine, CustomWeightDecaySGD, AdamW, RSquared
from network import HierarchicalConvEmbedding, TransformerBlock, AddPositionEmbedding, MLP, MixerBlock

In [ ]:
SAVE_PATH = "utkface_12k_split.npz" 
def load_dataset_if_exists(path=SAVE_PATH):
    """
    If the file exists, load it. Otherwise, return None.
    """
    if os.path.exists(path):
        data = np.load(path)
        print("✔ Detected cached data, loaded directly")
        return (data["x_train"], data["y_train"],
                data["x_test"], data["y_test"])
    return None

In [5]:
x_train, y_train, x_test, y_test = load_dataset_if_exists()

✔ 检测到缓存数据，已直接加载


In [ ]:
# ============================================================
# 32x32 分辨率：推荐的起步参数（经验值）
# ============================================================
def suggested_l2_eps_for_32x32():
    """
    For [0,1] normalized 32x32 images:
      - Light perturbation: eps ≈ 0.5 ~ 1.5
      - Medium strength: eps ≈ 2.0 ~ 3.0
      - Stronger perturbation: eps ≈ 4.0+ (may be noticeable)
    You can start with eps=2.0 for attack strength curves.
    """
    return {"light": (0.5, 1.5), "medium": (2.0, 3.0), "strong": (4.0, 6.0)}

In [7]:
def attack_models(model):
    attacker = PGDRegressionAdversaryL2(model, clip_min=0.0, clip_max=1.0)
    x_adv = attacker.generate_adversarial_dataset(x_train, y_train, eps=2.0, alpha=0.2, steps=20,
                                  batch_size=128,
                                  shuffle=True,
                                  random_start=True)

    return x_adv

In [8]:
NOISE_DIR = "./attack/"

In [11]:
def save_wb_per_layer(NOISE_DIR, N=5):
    for i in range(N):
        NOISE_I_DIR = NOISE_DIR + str(i)
        os.makedirs(NOISE_I_DIR, exist_ok=True)
        x_pgd_VGG= attack_models(model_VGG)
        model_VGG.evaluate(x_pgd_VGG,y_train)
        np.save(os.path.join(NOISE_I_DIR, "VGG_pgd.npy"), x_pgd_VGG, allow_pickle=True)
        x_pgd_Res = attack_models(model_Res)
        model_Res.evaluate(x_pgd_Res,y_train)
        np.save(os.path.join(NOISE_I_DIR, "Res_pgd.npy"), x_pgd_Res, allow_pickle=True)
        x_pgd_ViT = attack_models(model_ViT)
        model_ViT.evaluate(x_pgd_ViT,y_train)
        np.save(os.path.join(NOISE_I_DIR, "ViT_pgd.npy"), x_pgd_ViT, allow_pickle=True)
        x_pgd_Mix = attack_models(model_Mix)
        model_Mix.evaluate(x_pgd_Mix,y_train)
        np.save(os.path.join(NOISE_I_DIR, "Mix_pgd.npy"), x_pgd_Mix, allow_pickle=True)
    print("Saved all noise data into separate files.")

In [12]:
model_VGG=load_VGG()
model_Res=load_Res()
model_ViT=load_ViT()
model_Mix=load_Mix()
save_wb_per_layer(NOISE_DIR)

625/625 [==============================] - 11s 17ms/step - loss: 0.4432 - r_squared: -13.3795
Saved all noise data into separate files.
